# 01 — Searching OpenAlex for a claim

**Goal:** send one claim to the OpenAlex API, understand what comes back, and check whether the search ranking helps find evidence.

**Steps:** setup → search → explore the response → relevance score → which sentence matches the claim.

Devlog for this notebook: `01_openalex_search.devlog.md`

## 1. Setup
`requests` is the library for calling web APIs.

In [72]:
import requests  # sends HTTP requests to web APIs

The API key comes from a Windows **environment variable**

In [73]:
import os

# Read the key saved once with: setx OPENALEX_API_KEY "..."
api_key = os.environ.get("OPENALEX_API_KEY")
print("Key loaded:", api_key is not None)  

Key loaded: True


## 2. The claim
The sentence we want to find evidence for.

In [74]:
claim = "Auditory attention decoding can be decoded using EEG"

## 3. Search OpenAlex
Send the claim to the `/works` endpoint (works = papers).
- `search` — keyword search over title, abstract and full text
- `per_page` — how many papers to return (explained below)
- The key goes in a **header**, not the URL, so it can't leak into URLs or error messages.

In [75]:
url = "https://api.openalex.org/works"  # endpoint for papers ("works")
params = {
    "search": claim,   # what to search for
    "per_page": 10     # number of papers to return
}
headers = {"Authorization": f"Bearer {api_key}"}  # API key sent as a header
response = requests.get(url, params=params, headers=headers, timeout=30)  # give up after 30 s
print(response.status_code)  # 200 = OK, 429 = too many requests, 400 = bad parameters
data = response.json()       # JSON text -> Python dict
print(data.keys())

200
dict_keys(['meta', 'results', 'group_by'])


## 4. Explore the response
`data` has three parts: `meta` (info about the search), `results` (the papers) and `group_by` (only used when counting groups).

In [76]:
print(type(data))                # what kind of Python object is it?
print(data.keys())
print(data["meta"])              # information about the search itself

<class 'dict'>
dict_keys(['meta', 'results', 'group_by'])
{'count': 16765, 'db_response_time_ms': 286, 'page': 1, 'per_page': 10, 'groups_count': None, 'x_query': {'oql': 'works where full text has (Auditory attention decoding can be decoded using EEG)', 'oqo': {'get_rows': 'works', 'filter_rows': [{'column_id': 'fulltext.search', 'value': 'Auditory attention decoding can be decoded using EEG', 'operator': 'has'}]}, 'url': '/works?filter=fulltext.search:Auditory attention decoding can be decoded using EEG&per_page=10'}, 'cost_usd': 0.001}


### What is `per_page`?

`per_page` controls **how many papers OpenAlex returns in one response**.

A search can match thousands of papers. OpenAlex does not send them all at once
(the response would be huge and slow). Instead, it splits the results into **pages**,
like pages of Google results.

| Setting | Page 1 contains | Page 2 contains |
|---|---|---|
| `per_page=10` | papers 1–10 | papers 11–20 |
| `per_page=25` (default) | papers 1–25 | papers 26–50 |

- Minimum: 1, maximum: **100** per page
- Default if not set: **25**
- Choose which page to fetch with the `page` parameter:

```python
params = {"search": claim, "per_page": 10, "page": 2}   # papers 11–20
```

- The total number of matching papers is in `data["meta"]["count"]`.

In [77]:
# Check per_page worked: a misspelled parameter is silently ignored (default = 25)
print("Results returned:", len(data["results"]))
print("per_page according to meta:", data["meta"].get("per_page"))

Results returned: 10
per_page according to meta: 10


### Size and speed of one request

In [78]:
# response.headers = what OpenAlex sent back (safe to print).
# Never print response.request.headers: it contains the API key.
print("Size in bytes:", len(response.content))
print("Size in KB:", round(len(response.content) / 1024, 1))
print("Time taken:", response.elapsed)
print("Content type:", response.headers["Content-Type"])

Size in bytes: 329098
Size in KB: 321.4
Time taken: 0:00:01.182341
Content type: application/json


### What is inside one paper?
Each paper is a dict with many fields. This prints each field's name, type and a preview. `NoneType` = missing for this paper.

In [79]:
paper = data["results"][0]  # the top-ranked paper
print("Number of fields:", len(paper))

# One line per field: name, type, first 200 characters of the value
for key, value in paper.items():
    preview = str(value)[:200]
    print(f"{key:30} {type(value).__name__:6} {preview}")

Number of fields: 52
id                             str    https://openalex.org/W2158904676
doi                            str    https://doi.org/10.1093/cercor/bht355
title                          str    Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG
display_name                   str    Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG
relevance_score                float  1522.515
publication_year               int    2014
publication_date               str    2014-01-15
ids                            dict   {'openalex': 'https://openalex.org/W2158904676', 'doi': 'https://doi.org/10.1093/cercor/bht355', 'mag': '2158904676', 'pmid': 'https://pubmed.ncbi.nlm.nih.gov/24429136'}
language                       str    en
primary_location               dict   {'id': 'doi:10.1093/cercor/bht355', 'is_oa': True, 'landing_page_url': 'https://doi.org/10.1093/cercor/bht355', 'pdf_url': 'https://academic.oup.com

### Nested fields
Some fields hold lists or dicts. The abstract is **not plain text**: it's an *inverted index* (word → list of positions).

In [80]:
print(paper["authorships"][0].keys())      # a list of dicts, one per author
print(paper["primary_location"])            # where it was published
aii = paper["abstract_inverted_index"]
print(type(aii), "words:", len(aii) if aii else None)

dict_keys(['author_position', 'author', 'institutions', 'countries', 'is_corresponding', 'raw_author_name', 'raw_affiliation_strings', 'raw_orcid', 'affiliations'])
{'id': 'doi:10.1093/cercor/bht355', 'is_oa': True, 'landing_page_url': 'https://doi.org/10.1093/cercor/bht355', 'pdf_url': 'https://academic.oup.com/cercor/article-pdf/25/7/1697/14102659/bht355.pdf', 'source': {'id': 'https://openalex.org/S117898428', 'display_name': 'Cerebral Cortex', 'issn_l': '1047-3211', 'issn': ['1047-3211', '1460-2199'], 'is_oa': False, 'is_in_doaj': False, 'is_core': True, 'listed_in': ['cwts-core', 'doyens', 'jufo-3', 'ki-jl-2', 'medline', 'norway-2'], 'host_organization': 'https://openalex.org/P4310311648', 'host_organization_name': 'Oxford University Press', 'host_organization_lineage': ['https://openalex.org/P4310311648', 'https://openalex.org/P4310311647'], 'host_organization_lineage_names': ['Oxford University Press', 'University of Oxford'], 'type': 'journal'}, 'license': None, 'license_id': N

In [81]:
import json
# Pretty-print the whole paper (uncomment to view)
#print(json.dumps(paper, indent=2)[:3000])   # first 3000 characters

In [82]:
data["results"][0]["title"]  # title of the top-ranked paper

'Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG'

## Relevance score vs. evidence: what OpenAlex tells us (and what it doesn't)

### What `relevance_score` actually is
- A **search-ranking number** for the *whole paper*, computed by OpenAlex from:
  1. **Text similarity**: how well my search words match the title, abstract and full text
  2. **Citation count**: more-cited papers get a higher score
- Results are sorted by it (highest first) by default.
- Source: [OpenAlex searching guide](https://developers.openalex.org/guides/searching)

### Important: what it is **not**
- **Not a confidence score** that the paper supports my claim.
- **Not tied to a sentence.** OpenAlex does not return which sentence or words matched (no highlighting/snippets in the documented API).
- **Not comparable across searches.** There is no fixed scale; a score of 40 in one query means nothing for another.
- The match may come from the **full text**, which is not in the response, so I can't always see *why* a paper ranked high.

### Drawbacks for Evidentia
- **Citation bias:** a highly cited review that barely touches my claim can outrank a smaller study that tests it directly.
- **Keyword matching, not meaning:** matching words ≠ supporting the claim.
- **No stance:** a paper that *contradicts* the claim can score just as high as one that supports it.

### My first workaround: sentence-level word overlap
To see *which sentence* relates to the claim, I:
1. rebuild the abstract from `abstract_inverted_index`,
2. split it into sentences,
3. rank sentences by the number of (non-stop) words they share with the claim.

This is **my own heuristic**, not what OpenAlex computed.

### Limitations of the workaround
- **No stemming:** "decoded" and "decoding" count as different words.
- **Ignores meaning and negation:** "attention could *not* be decoded from EEG" shares many words with the claim but says the opposite.
- **Misses paraphrases:** strong evidence may use none of my words (e.g. "listeners' focus was tracked from cortical signals").
- **Abstract only:** the best evidence is often in the results section of the full text.

### What this points to next
1. Stemming / lemmatisation of words
2. Semantic similarity (embeddings) instead of word overlap
3. A model (LLM or trained classifier) that judges whether a sentence **supports, contradicts, or is unrelated to** the claim


In [83]:
# Relevance score and title of every result (results are sorted by score)
# Note: the { } around the values make Python sets, which is why the output shows braces
for n in range(len(data["results"])):
    print(f"relevance_score:-", {data["results"][n]["relevance_score"]}," : ","Paper Title :- ",{data["results"][n]["title"]})

relevance_score:- {1522.515}  :  Paper Title :-  {'Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG'}
relevance_score:- {963.96893}  :  Paper Title :-  {'MEG and EEG data analysis with MNE-Python'}
relevance_score:- {688.87744}  :  Paper Title :-  {'A Comparison of Regularization Methods in Forward and Backward Models for Auditory Attention Decoding'}
relevance_score:- {603.02356}  :  Paper Title :-  {'A review of classification algorithms for EEG-based brain–computer interfaces: a 10 year update'}
relevance_score:- {600.61865}  :  Paper Title :-  {'Decoding Attended Information in Short-term Memory: An EEG Study'}
relevance_score:- {599.5711}  :  Paper Title :-  {'STAnet: A Spatiotemporal Attention Network for Decoding Auditory Spatial Attention From EEG'}
relevance_score:- {592.35425}  :  Paper Title :-  {'Fast EEG-Based Decoding Of The Directional Focus Of Auditory Attention Using Common Spatial Patterns'}
relevance_score:- {590.1436}  :  Pa

## 5. Which sentence matches the claim?
My own simple heuristic (not from OpenAlex): rebuild each abstract, split it into sentences, and rank sentences by how many words they share with the claim. Limitations are listed in the section above.

In [84]:
import re  # regular expressions, for finding words and splitting sentences

def rebuild_abstract(aii):
    # Turn the inverted index {word: [positions]} back into normal text
    if not aii:
        return None
    positions = [(i, word) for word, idxs in aii.items() for i in idxs]
    return " ".join(word for i, word in sorted(positions))

# Common words to ignore when comparing
STOP = {"the", "a", "an", "of", "in", "on", "and", "or", "to", "is", "are",
        "be", "can", "using", "with", "for", "by", "from", "that", "this"}

def words(text):
    # Lowercase words in the text, minus stop words
    return set(re.findall(r"[a-z]+", text.lower())) - STOP

claim_words = words(claim)

for paper in data["results"][:5]:  # top 3 papers only
    abstract = rebuild_abstract(paper["abstract_inverted_index"])
    print("\n###", paper["display_name"], "| score:", paper["relevance_score"])
    if not abstract:
        print("   (no abstract)")
        continue
    sentences = re.split(r"(?<=[.!?])\s+", abstract)  # split after . ! ?
    # Sort sentences by number of words shared with the claim (most first)
    ranked = sorted(sentences, key=lambda s: len(claim_words & words(s)), reverse=True)
    for s in ranked[:2]:  # show the best 2 sentences
        shared = claim_words & words(s)
        print(f"   [{len(shared)} shared: {sorted(shared)}] {s}")


### Attentional Selection in a Cocktail Party Environment Can Be Decoded from Single-Trial EEG | score: 1522.515
   [2 shared: ['decoded', 'eeg']] Here we show that single-trial (≈60 s) unaveraged EEG data can be decoded to determine attentional selection in a naturalistic multispeaker environment.
   [2 shared: ['attention', 'eeg']] Furthermore, we show a significant correlation between our EEG-based measure of attention and performance on a high-level attention task.

### MEG and EEG data analysis with MNE-Python | score: 963.96893
   [1 shared: ['eeg']] Magnetoencephalography and electroencephalography (M/EEG) measure the weak electromagnetic signals generated by neuronal activity in the brain.
   [1 shared: ['eeg']] All algorithms and utility functions are implemented in a consistent manner with well-documented interfaces, enabling users to create M/EEG data analysis pipelines by writing Python scripts.

### A Comparison of Regularization Methods in Forward and Backward Models for